

## Load NAB time series

## Step 1 - Load ambient dataset and attach anomaly labels

In this step, the raw `ambient_temperature_system_failure.csv` file from the NAB
`realKnownCause` folder is loaded into a dataframe. The corresponding anomaly
timestamps are then retrieved from `combined_labels.json` and converted into a
binary `is_anomaly` column. The timestamp column is parsed as a proper datetime
type and the rows are sorted chronologically so that each record reflects the
correct order in time.


In [15]:
## Imports
import pandas as pd
import numpy as np

In [2]:
import os

# Path to Numeta Anomoly Benchmark (NAB) data folder
base_path = "../data/NAB-master/data/realKnownCause/"

# Load one example file
file_path = os.path.join(base_path, "ambient_temperature_system_failure.csv")
df = pd.read_csv(file_path)

df.head()


,timestamp,value
0,2013-07-04 00:00:00,69.880835
1,2013-07-04 01:00:00,71.220227
2,2013-07-04 02:00:00,70.877805
3,2013-07-04 03:00:00,68.959400
4,2013-07-04 04:00:00,69.283551


## Load Anomaly Labels (JSON)

In [3]:
import json  # to work with JSON files

# Tell Python where the labels file is
label_path = "../data/NAB-master/labels/combined_labels.json"

# Open the file and read its contents
with open(label_path, "r") as label_file:      # "r" = read mode, label_file = name for the opened file
    labels = json.load(label_file)             # turn the JSON into a Python dictionary called 'labels'

# 3. Shows the first 10 dataset keys names to confirm it loaded correctly
list(labels.keys())[:10]


['artificialNoAnomaly/art_daily_no_noise.csv',
 'artificialNoAnomaly/art_daily_perfect_square_wave.csv',
 'artificialNoAnomaly/art_daily_small_noise.csv',
 'artificialNoAnomaly/art_flatline.csv',
 'artificialNoAnomaly/art_noisy.csv',
 'artificialWithAnomaly/art_daily_flatmiddle.csv',
 'artificialWithAnomaly/art_daily_jumpsdown.csv',
 'artificialWithAnomaly/art_daily_jumpsup.csv',
 'artificialWithAnomaly/art_daily_nojump.csv',
 'artificialWithAnomaly/art_increase_spike_density.csv']

## Extract Labels for Selected Dataset

In [4]:
# Select the correct dataset key from the labels dictionary
dataset_key = "realKnownCause/ambient_temperature_system_failure.csv"

# Extract the list of anomaly timestamps for this dataset
label_times = labels[dataset_key]

# Display the timestamps
label_times

['2013-12-22 20:00:00', '2014-04-13 09:00:00']

## Add Labels to DataFrame

In [5]:
# Make sure the timestamp column is in datetime format
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Converting the label timestamps into datetime objects 
label_times_dt = pd.to_datetime(label_times)

# Create a new column: 1 if the timestamp is an anomaly, 0 otherwise
df["is_anomaly"] = df["timestamp"].isin(label_times_dt).astype(int)

# See the updated dataframe
df.head()


,timestamp,value,is_anomaly
0,2013-07-04 00:00:00,69.880835,0
1,2013-07-04 01:00:00,71.220227,0
2,2013-07-04 02:00:00,70.877805,0
3,2013-07-04 03:00:00,68.959400,0
4,2013-07-04 04:00:00,69.283551,0


## Class Balance

In [6]:
# Class balance for the is_anomaly column
anomaly_balance = (
    df["is_anomaly"]
      .value_counts()                            # counts for 0 and 1
      .rename(index={0: "Normal", 1: "Anomaly"}) # rename classes for readability
      .to_frame(name="Count")                    # make it a DataFrame with a 'Count' column
      .reset_index()                             # turn index into a normal column
      .rename(columns={"is_anomaly": "Class"})       
)

# Percentage column
anomaly_balance["Proportion (%)"] = (
    anomaly_balance["Count"] / len(df) * 100
).round(3)

anomaly_balance


,Class,Count,Proportion (%)
0,Normal,7265,99.972
1,Anomaly,2,0.028


## Step 2 - Standardise core columns (`time`, `value`, `is_anomaly`, `case_study`)

In [7]:
# Start from the dataframe we already built:
# df has columns: timestamp, value, is_anomaly

#  Rename timestamp -> time
df = df.rename(columns={"timestamp": "time"})

# Add a case_study identifier
df["case_study"] = "ambient"

# Quick check
df[["time", "value", "is_anomaly", "case_study"]].head()


,time,value,is_anomaly,case_study
0,2013-07-04 00:00:00,69.880835,0,ambient
1,2013-07-04 01:00:00,71.220227,0,ambient
2,2013-07-04 02:00:00,70.877805,0,ambient
3,2013-07-04 03:00:00,68.959400,0,ambient
4,2013-07-04 04:00:00,69.283551,0,ambient


## Step 3 - Convert `value` from Fahrenheit to Celsius 


In [8]:
# Convert the main `value` column from Fahrenheit to Celsius in place
df["value"] = (df["value"] - 32.0) * 5.0 / 9.0

# Quick sanity check
df[["time", "value", "is_anomaly", "case_study"]].head()
#df["value"].describe()


,time,value,is_anomaly,case_study
0,2013-07-04 00:00:00,21.044908,0,ambient
1,2013-07-04 01:00:00,21.789015,0,ambient
2,2013-07-04 02:00:00,21.598781,0,ambient
3,2013-07-04 03:00:00,20.533000,0,ambient
4,2013-07-04 04:00:00,20.713084,0,ambient


### Temperature Summary Table 

In [9]:
# Raw summary stats for Celsius values
value_summary = df["value"].describe()

# Turn into a one-column DataFrame and clean up labels
value_summary = (
    value_summary
    .to_frame(name="Ambient Temperature (°C)")
    .rename(index={
        "count": "Count",
        "mean": "Mean",
        "std":  "Std. deviation",
        "min":  "Min",
        "25%":  "Q1 (25%)",
        "50%":  "Median (50%) ",
        "75%":  "Q3 (75%)",
        "max":  "Max",
    })
    .round(2)
)

value_summary

,Ambient Temperature (°C)
Count,7267.00
Mean,21.80
Std. deviation,2.36
Min,14.14
Q1 (25%),20.21
Median (50%),22.14
Q3 (75%),23.57
Max,30.12


## Step 4 - Create basic time-based features

In [10]:
# Create time-based features
df["hour_of_day"] = df["time"].dt.hour
df["day_of_week"] = df["time"].dt.dayofweek  # Monday=0, Sunday=6
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# Quick check of new columns
df[["time", "hour_of_day", "day_of_week", "is_weekend"]].head()


,time,hour_of_day,day_of_week,is_weekend
0,2013-07-04 00:00:00,0,3,0
1,2013-07-04 01:00:00,1,3,0
2,2013-07-04 02:00:00,2,3,0
3,2013-07-04 03:00:00,3,3,0
4,2013-07-04 04:00:00,4,3,0


This step derives simple calendar features from the `time` column to provide
additional structure for later analysis and modelling. The timestamp is broken
down into:

- `hour_of_day`: the hour of the day (0–23),
- `day_of_week`: the day index (0 = Monday, …, 6 = Sunday),
- `is_weekend`: an indicator variable that is 1 for Saturday and Sunday, and 0
  for weekdays.

These features do not change the underlying temperature values, but they allow
later models and visualisations to capture patterns that depend on time of day
or weekday/weekend effects.


## Step 5 - Reorder columns for a clean schema

In [19]:
# Desired column order
column_order = [
    "time",
    "value",
    "is_anomaly",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "case_study",
]

# Reorder the dataframe
df = df[column_order]

df.head()


,time,value,is_anomaly,hour_of_day,day_of_week,is_weekend,case_study
0,2013-07-04 00:00:00,21.044908,0,0,3,0,ambient
1,2013-07-04 01:00:00,21.789015,0,1,3,0,ambient
2,2013-07-04 02:00:00,21.598781,0,2,3,0,ambient
3,2013-07-04 03:00:00,20.533000,0,3,3,0,ambient
4,2013-07-04 04:00:00,20.713084,0,4,3,0,ambient


## Step 6 - Inspect time span and anomaly distribution (before data split)

In [24]:
# Overall time span
start_time = df["time"].min()
end_time = df["time"].max()
n_rows = len(df)
duration = end_time - start_time

time_span_summary = pd.DataFrame(
    [
        ("Start time", start_time),
        ("End time", end_time),
        ("Number of rows", n_rows),
        ("Total duration (days)", duration.days),
    ],
    columns=["Statistic", "Value"],
)

time_span_summary


,Statistic,Value
0,Start time,2013-07-04 00:00:00
1,End time,2014-05-28 15:00:00
2,Number of rows,7267
3,Total duration (days),328


In [25]:
# Summary table for the ambient anomalies
anomaly_summary = (
    df.loc[df["is_anomaly"] == 1, ["time", "value"]]
      .reset_index()  # keep original row index for reference
      .rename(columns={
          "index": "Row index",
          "time": "Anomaly time",
          "value": "Temperature (°C)",
      })
      .round({"Temperature (°C)": 2})
)

anomaly_summary


,Row index,Anomaly time,Temperature (°C)
0,3721,2013-12-22 20:00:00,30.11
1,6180,2014-04-13 09:00:00,14.14


These summaries provide a clear view of how long the series runs and how many
observations it contains, as well as where the rare anomaly events occur in
time. This context will guide the choice of chronological split points in the
next step.


### Quick internal check of sampling interval




In [26]:
# Compute differences between consecutive timestamps
time_difference = df["time"].diff().dropna()

# Identify the most common time difference
most_common_difference = time_difference.value_counts().idxmax()

most_common_difference


Timedelta('0 days 01:00:00')

A detailed analysis of the sampling interval and any gaps in the series is
already provided in the Data Overview notebook for the ambient dataset.
Here, we only perform a minimal internal check to confirm that the dominant
interval between observations is one hour, consistent with the agreed
assumptions for this case study.

## Step 7 - Define a first draft of train /validation /test splits

### Define training and validation boundaries using anomaly times


In [35]:
# Extract anomaly times from the anomaly_summary table
first_anomaly_time = anomaly_summary.loc[0, "Anomaly time"]
second_anomaly_time = anomaly_summary.loc[1, "Anomaly time"]

# Place split boundaries 7 days before each anomaly
training_end_time = first_anomaly_time - pd.Timedelta(days=7)
validation_end_time = second_anomaly_time - pd.Timedelta(days=7)

# Summary table of the chosen boundaries and anomaly times
split_boundaries = pd.DataFrame(
    [
        ("Training end time", training_end_time),
        ("Validation end time", validation_end_time),
        ("First anomaly time", first_anomaly_time),
        ("Second anomaly time", second_anomaly_time),
    ],
    columns=["Description", "Time"],
)

split_boundaries


,Description,Time
0,Training end time,2013-12-15 20:00:00
1,Validation end time,2014-04-06 09:00:00
2,First anomaly time,2013-12-22 20:00:00
3,Second anomaly time,2014-04-13 09:00:00


In this step, the timestamps of the two labelled anomalies are used to define
chronological boundaries for the training and validation segments.

The anomaly times are
stored as `first_anomaly_time` and `second_anomaly_time`. 
The end of the
training period is then set to seven days before the first anomaly, and the end
of the validation period is set to seven days before the second anomaly. This
one-week buffer is intended to keep the training data focused on clearly normal
behaviour, while reserving the neighbourhood of each anomaly for validation and
test evaluation.

The table reports the resulting training and validation end
times alongside the original anomaly times. This provides a transparent record
of how the temporal split points were chosen and can be cited directly in the
methodology or data preparation section of the thesis.


### Assign split labels and summarise train/validation/test segments


In [36]:
# Helper function to assign each timestamp to a split
def assign_split(time_value):
    if time_value <= training_end_time:
        return "train"
    elif time_value <= validation_end_time:
        return "validation"
    else:
        return "test"

# Add the split label column to the dataframe
df["split"] = df["time"].apply(assign_split)

# Total number of rows for proportion calculation
total_rows = len(df)

# Summarise rows and anomalies per split
split_summary = (
    df.groupby("split")
      .agg(
          Start_time=("time", "min"),
          End_time=("time", "max"),
          Rows=("time", "size"),
          Anomalies=("is_anomaly", "sum"),
      )
      .reset_index()
)

# Enforce the desired ordering: train, validation, test
split_order = ["train", "validation", "test"]
split_summary["split"] = pd.Categorical(
    split_summary["split"],
    categories=split_order,
    ordered=True,
)
split_summary = split_summary.sort_values("split")

# Add a proportion column (percentage of the full dataset)
split_summary["Proportion (%)"] = (
    split_summary["Rows"] / total_rows * 100
).round(2)

split_summary


,split,Start_time,End_time,Rows,Anomalies,Proportion (%)
1,train,2013-07-04 00:00:00,2013-12-15 20:00:00,3554,0,48.91
2,validation,2013-12-15 21:00:00,2014-04-03 09:00:00,2560,1,35.23
0,test,2014-04-10 15:00:00,2014-05-28 15:00:00,1153,1,15.87


In [28]:
# Helper function to assign each timestamp to a split
def assign_split(timestamp):
    if timestamp <= train_end_time:
        return "train"
    elif timestamp <= val_end_time:
        return "validation"
    else:
        return "test"

# Apply the function row by row
df["split"] = df["time"].apply(assign_split)

# Summarise rows and anomalies per split
split_summary = (
    df.groupby("split")
      .agg(
          Start_time=("time", "min"),
          End_time=("time", "max"),
          Rows=("time", "size"),
          Anomalies=("is_anomaly", "sum"),
      )
      .reset_index()
)

split_summary


,split,Start_time,End_time,Rows,Anomalies
0,test,2014-04-10 15:00:00,2014-05-28 15:00:00,1153,1
1,train,2013-07-04 00:00:00,2013-12-15 20:00:00,3554,0
2,validation,2013-12-15 21:00:00,2014-04-03 09:00:00,2560,1


## Reorder columns for a clean schema

In [37]:
# Desired column order
column_order = [
    "time",
    "value",
    "is_anomaly",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "split",
    "case_study",
]

# Reorder the dataframe
df = df[column_order]

df.head()


,time,value,is_anomaly,hour_of_day,day_of_week,is_weekend,split,case_study
0,2013-07-04 00:00:00,21.044908,0,0,3,0,train,ambient
1,2013-07-04 01:00:00,21.789015,0,1,3,0,train,ambient
2,2013-07-04 02:00:00,21.598781,0,2,3,0,train,ambient
3,2013-07-04 03:00:00,20.533000,0,3,3,0,train,ambient
4,2013-07-04 04:00:00,20.713084,0,4,3,0,train,ambient
